In [ ]:
import os
import torch
import torch.nn as nn
import torch.utils.data as Data
import torchvision
import torch.nn.functional as F
import numpy as np
learning_rate = 1e-4
keep_prob_rate = 0.7 #
max_epoch = 3
BATCH_SIZE = 50

from tensorflow.keras.datasets import mnist as keras_mnist
import struct, gzip, io

def _ensure_mnist_files(root):
    """将 keras 缓存的 MNIST 转换为 torchvision 所需的格式"""
    raw_dir = os.path.join(root, 'MNIST', 'raw')
    # 如果已有处理好的数据就跳过
    processed_dir = os.path.join(root, 'MNIST', 'processed')
    if os.path.exists(processed_dir) and os.listdir(processed_dir):
        return
    if os.path.exists(raw_dir) and len(os.listdir(raw_dir)) >= 4:
        return

    os.makedirs(raw_dir, exist_ok=True)
    (x_train, y_train), (x_test, y_test) = keras_mnist.load_data()

    def write_images(filepath, images):
        with gzip.open(filepath, 'wb') as f:
            n, rows, cols = images.shape
            f.write(struct.pack('>IIII', 2051, n, rows, cols))
            f.write(images.tobytes())

    def write_labels(filepath, labels):
        with gzip.open(filepath, 'wb') as f:
            f.write(struct.pack('>II', 2049, len(labels)))
            f.write(labels.astype(np.uint8).tobytes())

    write_images(os.path.join(raw_dir, 'train-images-idx3-ubyte.gz'), x_train)
    write_labels(os.path.join(raw_dir, 'train-labels-idx1-ubyte.gz'), y_train)
    write_images(os.path.join(raw_dir, 't10k-images-idx3-ubyte.gz'), x_test)
    write_labels(os.path.join(raw_dir, 't10k-labels-idx1-ubyte.gz'), y_test)

_ensure_mnist_files('./mnist/')

train_data = torchvision.datasets.MNIST(root='./mnist/',train=True, transform=torchvision.transforms.ToTensor(), download=False)
train_loader = Data.DataLoader(dataset = train_data ,batch_size= BATCH_SIZE ,shuffle= True)

test_data = torchvision.datasets.MNIST(root = './mnist/',train = False, transform=torchvision.transforms.ToTensor(), download=False)
test_x = torch.unsqueeze(test_data.data, dim=1).type(torch.FloatTensor)[:500]/255.
test_y = test_data.targets[:500].numpy()

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                # patch 7 * 7
                in_channels=1,
                out_channels=32,
                kernel_size=7,
                stride=1,
                padding=3,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),

        )
        self.out1 = nn.Linear( 7*7*64 , 1024 , bias= True)   # full connection layer one

        self.dropout = nn.Dropout(keep_prob_rate)
        self.out2 = nn.Linear(1024,10,bias=True)



    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)  # flatten the output of conv2 to (batch_size, 64 * 7 * 7)
        out1 = self.out1(x)
        out1 = F.relu(out1)
        out1 = self.dropout(out1)
        out2 = self.out2(out1)
        return out2


def test(cnn):
    global prediction
    with torch.no_grad():
        y_pre = cnn(test_x)
    _,pre_index= torch.max(y_pre,1)
    pre_index= pre_index.view(-1)
    prediction = pre_index.data.numpy()
    correct  = np.sum(prediction == test_y)
    return correct / 500.0


def train(cnn):
    optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate )
    loss_func = nn.CrossEntropyLoss()
    for epoch in range(max_epoch):
        for step, (x_, y_) in enumerate(train_loader):
            output = cnn(x_)  
            loss = loss_func(output,y_)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if step != 0 and step % 20 ==0:
                print("=" * 10,step,"="*5,"="*5, "test accuracy is ",test(cnn) ,"=" * 10 )

if __name__ == '__main__':
    cnn = CNN()
    train(cnn)


========== 20 ===== ===== test accuracy is  0.386 ==========
========== 40 ===== ===== test accuracy is  0.618 ==========
========== 60 ===== ===== test accuracy is  0.654 ==========
========== 80 ===== ===== test accuracy is  0.72 ==========
========== 100 ===== ===== test accuracy is  0.73 ==========
========== 120 ===== ===== test accuracy is  0.784 ==========
========== 140 ===== ===== test accuracy is  0.802 ==========
========== 160 ===== ===== test accuracy is  0.846 ==========
========== 180 ===== ===== test accuracy is  0.838 ==========
========== 200 ===== ===== test accuracy is  0.852 ==========
========== 220 ===== ===== test accuracy is  0.862 ==========
========== 240 ===== ===== test accuracy is  0.89 ==========
========== 260 ===== ===== test accuracy is  0.876 ==========
========== 280 ===== ===== test accuracy is  0.888 ==========
========== 300 ===== ===== test accuracy is  0.89 ==========
========== 320 ===== ===== test accuracy is  0.89 ==========
========== 340 ==